# Notebook 3: Explainable AI (XAI) -- SHAP on tabular data

## Course: Big Data Analytics
## University of Milano-Bicocca
## Degree Programme: Economics, Data Analysis and Management

---

### Introduction

In **Notebook 1** we trained a model that decides on **mortgage applications** and opened it with LIME. Here we **train the same identical model again** and open it with **SHAP** (*SHapley Additive exPlanations*, Lundberg & Lee, 2017), built on the **Shapley value** of cooperative game theory (Lloyd Shapley, 1953; Shapley received the Nobel Prize in Economics in 2012).

**What SHAP adds to LIME**
- contributions that **add up exactly** to the prediction;
- a **global** view, obtained by aggregating many local explanations.

> 📚 **The series**: 1. LIME · tabular → 2. LIME · text → **3. SHAP · tabular** (this notebook) → 4. SHAP · text. Notebooks 1 and 3 explain **the same mortgage model**.

# NINE FEATURES WORK TOGETHER TO PRODUCE ONE DECISION. HOW DO WE SPLIT THE CREDIT FAIRLY AMONG THEM?

---

In [ ]:
# ==========================================
# 1. SETUP: INSTALL AND IMPORT
# ==========================================

# shap may already be installed on Colab; lime is used once, to compare the two methods
!pip install shap lime --quiet

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

import shap
from lime.lime_tabular import LimeTabularExplainer

warnings.filterwarnings("ignore")    # keep the output clean
np.random.seed(42)                   # reproducibility

# Colors of our own charts (colorblind-safe palette)
BLUE, ORANGE, RED, GRAY = "#2a78d6", "#eb6834", "#e34948", "#898781"
INK, GRID = "#52514e", "#e1e0d9"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.max_colwidth", 150)

print(f"✅ Setup complete (shap {shap.__version__})")

# 🏗️ Part 1 — Train the black box (the same as Notebook 1)

We **repeat the training of Notebook 1**, step by step: same data, same split, same random seeds → **the same identical model**. The *fingerprint* printed at the end of this part must be identical to the one printed in Notebook 1.

## 1.1 🏦 The data: mortgage applications

The **Boston HMDA** data (Munnell et al., 1996; also used in Stock & Watson's *Introduction to Econometrics*) contains **2,380 real mortgage applications** collected by the Federal Reserve Bank of Boston. We use 9 applicant characteristics:

| Feature | Meaning |
| :--- | :--- |
| `pirat` | debt payments / income (0.33 = 33% of the income goes to debt) |
| `hirat` | housing expenses / income |
| `lvrat` | loan / property value (*loan-to-value*) |
| `chist` | consumer credit history, from 1 (best) to 6 (worst) |
| `mhist` | mortgage credit history, from 1 (best) to 4 (worst) |
| `phist` | public bad credit record (e.g. bankruptcy)? 1 = yes |
| `selfemp` | self-employed? 1 = yes |
| `single` | single? 1 = yes |
| `afam` | African-American applicant? 1 = yes ⚠️ **protected attribute** |

**Target**: `deny` = 1 if the mortgage was denied.

In [ ]:
# ==========================================
# 2. LOAD THE MORTGAGE DATA
# ==========================================

URL_HMDA = "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/AER/HMDA.csv"
hmda = pd.read_csv(URL_HMDA)

# The 9 features we use. The file has a few more: e.g. 'insurance' (= a separate insurer
# refused the applicant) is almost a copy of the outcome, so we leave it out.
FEATURES = ['pirat', 'hirat', 'lvrat', 'chist', 'mhist', 'phist', 'selfemp', 'single', 'afam']

# yes/no columns -> 1/0
for col in ['deny', 'phist', 'selfemp', 'single', 'afam']:
    hmda[col] = (hmda[col] == 'yes').astype(int)

X = hmda[FEATURES]
y = hmda['deny']

print(f"{len(hmda)} applications | denied: {y.mean():.1%}")
X.head()

## 1.2 🌲 The black box: a Random Forest

A forest of 100 decision trees that vote together: accurate, but impossible to read.

In [ ]:
# ==========================================
# 3. TRAIN THE MORTGAGE BLACK BOX (RANDOM FOREST)
# ==========================================

# 75% train / 25% test, same denial rate in both sets (stratify), fixed seed
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
X_test, y_test = X_test.reset_index(drop=True), y_test.reset_index(drop=True)  # applicants #0, #1, ...

def train_mortgage_model(X, y):
    # Our black box: a forest of 100 decision trees that vote together
    return RandomForestClassifier(n_estimators=100, min_samples_leaf=10, random_state=42).fit(X, y)

mortgage_model = train_mortgage_model(X_train, y_train)
p_deny = mortgage_model.predict_proba(X_test)[:, 1]      # P(deny) of every test applicant

auc = roc_auc_score(y_test, p_deny)
print(f"Accuracy: {accuracy_score(y_test, p_deny > 0.5):.3f}  (careful: approving everybody already gives {1 - y_test.mean():.3f})")
print(f"ROC-AUC:  {auc:.4f}  (0.5 = random ranking, 1 = perfect ranking)")
print(f"\n🔑 Fingerprint of the model: AUC = {auc:.4f} | P(deny) of applicant #112 = {p_deny[112]:.3f}")
print("   (identical in Notebooks 1 and 3: same data, same code, same seeds -> same model)")

✅ **Same fingerprint as Notebook 1 → same model.** From now on, only the explainer changes.

# 🔍 Part 2 — How SHAP works

## 2.1 A cooperative game

| Game theory | Machine learning |
| :--- | :--- |
| players | features (`pirat`, `lvrat`, ..., `afam`) |
| a team $S$ | the features whose value we **know** |
| the payout of the team, $v(S)$ | the prediction when only the features in $S$ are known |
| the total to split | $f(x) - E[f(X)]$: prediction for this applicant − prediction for an "average" applicant |

**How do we predict when only some features are known?** The known features take the applicant's values, the unknown ones take the values of real applicants (the **background** $B$), and we average:

$$v(S) \;=\; \frac{1}{|B|}\sum_{b \in B} f\bigl(x_S,\; b_{\bar S}\bigr)$$

With no feature known, $v(\varnothing) = E[f(X)]$ is the **base value**; with all features known, $v(\text{all}) = f(x)$ is the prediction.

## 2.2 Order matters... so average over all orders

Let the players join the team **one at a time**: the *marginal contribution* of a player is how much $v$ changes when it joins. But it depends on **who joined before** (players interact). Shapley's solution is to **average the marginal contribution over all the possible orders**:

$$\phi_j \;=\; \frac{1}{n!}\sum_{\text{orders } R}\Bigl[\, v\bigl(P_j^R \cup \{j\}\bigr) - v\bigl(P_j^R\bigr) \Bigr]$$

where $P_j^R$ is the set of players that come **before** $j$ in the order $R$, and $n$ is the number of players.

## 2.3 Why is this split "fair"?

| Property | Meaning |
| :--- | :--- |
| **Efficiency** | $v(\varnothing) + \sum_j \phi_j = f(x)$: the contributions add up **exactly** to the prediction |
| **Symmetry** | two players that contribute the same to every team get the same value |
| **Dummy** | a player that never changes the prediction gets $0$ |
| **Additivity** | for a model made of a sum of models, the values add up |

Shapley proved that his value is the **only** split with all four properties.

## 2.4 🔧 SHAP by hand: applicant #112

Applicant #112 was **denied** (P(deny) ≈ 0.52), and in Notebook 1 LIME showed that race was among the reasons. Let's split this prediction with Shapley's recipe.

In [ ]:
# ==========================================
# 4. THE VALUE OF A TEAM v(S)
# ==========================================

APPLICANT = 112
applicant = X_test.iloc[APPLICANT]
background = X_train.sample(100, random_state=42)   # 100 real applicants = "the average applicant"

cache = {}
def v(S):
    # Average P(deny) when the features in S take the applicant's values
    # and all the other features take the values of the background applicants.
    key = frozenset(S)
    if key not in cache:
        Z = background.copy()
        for f in S:
            Z[f] = applicant[f]
        cache[key] = mortgage_model.predict_proba(Z)[:, 1].mean()
    return cache[key]

print(f"v(nothing known)  = {v([]):.3f}   <- base value E[f(X)]")
print(f"v(only 'phist')   = {v(['phist']):.3f}   <- we only know the public bad credit record")
print(f"v(all 9 features) = {v(FEATURES):.3f}   <- the model's prediction f(x) = {p_deny[APPLICANT]:.3f}")

In [ ]:
# ==========================================
# 5. TWO ORDERS, TWO DIFFERENT STORIES
# ==========================================

def contributions(order):
    # Marginal contribution of each feature when the features join in this order
    contrib, team = {}, []
    for f in order:
        before = v(team)
        team = team + [f]
        contrib[f] = v(team) - before
    return pd.Series(contrib)[FEATURES]

orders = pd.DataFrame({"order A": contributions(FEATURES),          # pirat, hirat, ..., afam
                       "order B": contributions(FEATURES[::-1])})    # afam, single, ..., pirat
print("order A = pirat first ... afam last  |  order B = afam first ... pirat last\n")
print(orders.round(3), "\n")
print("Sum of the contributions:", orders.sum().round(3).tolist(), f"= f(x) - base value = {p_deny[APPLICANT] - v([]):.3f}")

ax = orders.plot.barh(figsize=(8, 4.2), color=[BLUE, ORANGE], width=0.7)
ax.axvline(0, color=INK, lw=0.8)
ax.invert_yaxis()
ax.set_xlabel("marginal contribution to P(deny)")
ax.set_title(f"Applicant #{APPLICANT}: the same feature, two different contributions", loc="left")
ax.legend(frameon=False, loc="lower right")
ax.xaxis.grid(True, color=GRID, lw=0.8); ax.set_axisbelow(True)
plt.show()

# QUESTION: BOTH ORDERS ADD UP TO THE SAME TOTAL, BUT `phist` GETS VERY DIFFERENT CREDIT. WHICH NUMBER IS THE "TRUE" ONE?

**Neither**: features interact, so each order tells a partial story. Shapley's answer is to average over **all** orders. With 9 features there are 9! = 362,880 orders, so we average over 1,000 random ones (the 2⁹ = 512 possible teams are computed once and cached).

In [ ]:
# ==========================================
# 6. AVERAGE OVER 1,000 RANDOM ORDERS = SHAPLEY VALUES
# ==========================================

rng = np.random.RandomState(0)
all_orders = pd.DataFrame([contributions(list(rng.permutation(FEATURES))) for _ in range(1000)])
shap_by_hand = all_orders.mean()
print(f"Distinct teams evaluated: {len(cache)} (at most 2^9 = {2**9})\n")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
for ax, f in zip(axes, ["phist", "afam"]):
    ax.hist(all_orders[f], bins=30, color=BLUE, alpha=0.85)
    ax.axvline(shap_by_hand[f], color=INK, lw=2)
    ax.set_title(f"'{f}': Shapley value = mean = {shap_by_hand[f]:+.3f}", loc="left")
    ax.set_xlabel("marginal contribution in one order")
axes[0].set_ylabel("number of orders")
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# 7. THE SAME WITH THE shap LIBRARY
# ==========================================

def predict_deny(data):
    # shap passes a numpy array: we give the model its column names back
    return mortgage_model.predict_proba(pd.DataFrame(data, columns=FEATURES))[:, 1]

# Same background, all 2^9 teams, exact Shapley values
explainer = shap.Explainer(predict_deny, background, algorithm="exact")
sv_applicant = explainer(X_test.iloc[[APPLICANT]])

comparison = pd.DataFrame({"by hand (1,000 orders)": shap_by_hand,
                           "shap library (exact)": sv_applicant.values[0]}, index=FEATURES)
print(comparison.round(3), "\n")

base = sv_applicant.base_values[0]
total = base + sv_applicant.values[0].sum()
print(f"✅ Efficiency: base value {base:.3f} + sum of SHAP values {sv_applicant.values[0].sum():+.3f} "
      f"= {total:.3f} = model's P(deny) {p_deny[APPLICANT]:.3f}")

### 💡 From the mechanism to the practice

| Because SHAP... | ...it can | ...but |
| :--- | :--- | :--- |
| averages over **all orders** (all $2^n$ teams) | share interactions **fairly** among the features | the exact computation explodes with many features |
| satisfies **efficiency** | express every effect in **probability points** that add up exactly to $f(x)$ | — |
| compares with a **background** | answer *"why this applicant, compared with the average one?"* | a different background gives different values |
| fills unknown features with **real values** of other applicants | work with **any** model | correlated features can create unrealistic combinations |
| gives **consistent** local values | build **global** views by aggregating them (Part 3) | important ≠ causal |

# 🏦 Part 3 — SHAP on the mortgage model

## 3.1 Local view: the waterfall plot

We now explain 300 test applicants at once (the same exact computation, ≈ 30 seconds).

In [ ]:
# ==========================================
# 8. EXPLAIN 300 TEST APPLICANTS
# ==========================================

N_EXPLAIN = 300                       # applicants #0 ... #299 (they include #112)
sv = explainer(X_test.iloc[:N_EXPLAIN])
print(f"SHAP values: {sv.values.shape[0]} applicants x {sv.values.shape[1]} features")

In [ ]:
# ==========================================
# 9. WATERFALL PLOT OF APPLICANT #112
# ==========================================

shap.plots.waterfall(sv[APPLICANT])

**How to read it**: start from the bottom, $E[f(X)]$ = the average P(deny) of the background applicants. Each bar moves the prediction: **red** toward denial, **blue** toward approval. At the top, $f(x)$ is the model's prediction. All numbers are **probability points** and they add up exactly.

## 3.2 Same model, same applicant: LIME vs SHAP

In [ ]:
# ==========================================
# 10. LIME vs SHAP FOR APPLICANT #112
# ==========================================

binary_idx = [FEATURES.index(f) for f in ['phist', 'selfemp', 'single', 'afam']]

def predict_mortgage(data):
    # LIME passes a numpy array: we give the model its column names back
    return mortgage_model.predict_proba(pd.DataFrame(data, columns=FEATURES))

tab_explainer = LimeTabularExplainer(
    X_train.values, feature_names=FEATURES, class_names=['approved', 'denied'],
    categorical_features=binary_idx, categorical_names={i: ['no', 'yes'] for i in binary_idx},
    random_state=42)

exp_lime = tab_explainer.explain_instance(applicant.values, predict_mortgage, num_features=9, num_samples=5000)
lime_weights = pd.Series({FEATURES[i]: w for i, w in exp_lime.as_map()[1]})

both = pd.DataFrame({"LIME weight": lime_weights,
                     "SHAP value": pd.Series(sv[APPLICANT].values, index=FEATURES)}).loc[FEATURES]
ax = both.plot.barh(figsize=(8, 4.2), color=[BLUE, ORANGE], width=0.7)
ax.axvline(0, color=INK, lw=0.8)
ax.invert_yaxis()
ax.set_xlabel("effect on P(deny)")
ax.set_title(f"Applicant #{APPLICANT}: two explanations of the same model", loc="left")
ax.legend(frameon=False, loc="lower right")
ax.xaxis.grid(True, color=GRID, lw=0.8); ax.set_axisbelow(True)
plt.show()

print(f"Sum of LIME weights: {both['LIME weight'].sum():+.3f}  (no particular meaning)")
print(f"Sum of SHAP values:  {both['SHAP value'].sum():+.3f}  = f(x) - base value = {p_deny[APPLICANT] - base:+.3f}")

💡 The two methods tell the **same main story** (public record, credit history, loan-to-value, and race pushing toward denial), but they measure different things:
- **LIME**: the slopes of a *local linear approximation* ("what if this feature were in another range?"); the weights do not add up to anything.
- **SHAP**: a *fair split* of the distance from the average applicant; the values add up **exactly** to $f(x) - E[f(X)]$.

# QUESTION: THE BANK MUST EXPLAIN THE DENIAL TO APPLICANT #112. WOULD YOU USE THE LIME OR THE SHAP EXPLANATION? AND FOR A REGULATOR?

## 3.3 Global view: from 300 local explanations to the whole model

In [ ]:
# ==========================================
# 11. BEESWARM PLOT
# ==========================================

shap.plots.beeswarm(sv)

**How to read it**: one dot = one applicant. Horizontal position = SHAP value (right = toward denial). Color = the value of the feature (red = high, blue = low). Features are sorted by importance. For example, a public bad credit record (`phist = 1`, red) always pushes toward denial, while a low loan-to-value (`lvrat`, blue) pushes toward approval.

In [ ]:
# ==========================================
# 12. GLOBAL IMPORTANCE (MEAN |SHAP|) AND DEPENDENCE PLOT
# ==========================================

shap.plots.bar(sv)                      # average size of the contribution of each feature
shap.plots.scatter(sv[:, "pirat"])      # the rule learned for the debt-payments-to-income ratio

**Dependence plot**: each dot is an applicant; it shows *how* the contribution of `pirat` changes with its value. Below ≈ 0.4 (40% of the income spent on debt) the ratio barely matters; above it, the risk of denial rises steeply. This is the rule the forest has learned.

## 3.4 ⚖️ Fairness audit: how much does race add?

In [ ]:
# ==========================================
# 13. THE CONTRIBUTION OF RACE
# ==========================================

shap.plots.scatter(sv[:, "afam"])

is_afam = X_test["afam"].iloc[:N_EXPLAIN].values == 1
afam_shap = sv[:, "afam"].values[is_afam]
print(f"African-American applicants: {is_afam.sum()} | SHAP value of 'afam' > 0 for {np.mean(afam_shap > 0):.0%} of them")
print(f"On average race adds {afam_shap.mean():+.3f} to their P(deny) (min {afam_shap.min():+.3f}, max {afam_shap.max():+.3f})")

🚨 For **every** African-American applicant of our sample, the model adds a positive contribution to P(deny) **only because of race**, and SHAP measures it in probability points.

Efficiency gives us more: since the contributions add up, we can **split the gap** in predicted risk between the two groups into the contributions of the single features.

In [ ]:
# ==========================================
# 14. WHERE DOES THE GAP BETWEEN THE GROUPS COME FROM?
# ==========================================

def gap_by_feature(shap_values, names):
    # (mean SHAP of African-American applicants) - (mean SHAP of the others), feature by feature
    values = pd.DataFrame(shap_values.values, columns=names)
    return values[is_afam].mean() - values[~is_afam].mean()

gap = gap_by_feature(sv, FEATURES)
pred_gap = p_deny[:N_EXPLAIN][is_afam].mean() - p_deny[:N_EXPLAIN][~is_afam].mean()

fig, ax = plt.subplots(figsize=(8, 3.8))
order = gap.sort_values().index
bars = ax.barh(order, gap[order], height=0.55, color=[ORANGE if f == "afam" else BLUE for f in order])
ax.bar_label(bars, labels=[f"{v:+.3f}" for v in gap[order]], padding=3, color=INK, fontsize=8)
ax.axvline(0, color=INK, lw=0.8)
ax.set_xlim(min(-0.015, gap.min() - 0.01), gap.max() * 1.2)
ax.set_xlabel("contribution to the gap in P(deny)")
ax.set_title(f"Gap in predicted P(deny), African-American vs other applicants: {pred_gap:+.3f}", loc="left")
ax.xaxis.grid(True, color=GRID, lw=0.8); ax.set_axisbelow(True)
plt.show()
print(f"Sum of the bars: {gap.sum():+.3f} = gap in the average predictions {pred_gap:+.3f} ✅")

💡 Race itself explains **part** of the gap (orange bar); the rest comes from features that differ between the groups in the data (loan-to-value, credit history, ...).

# QUESTION: RACE EXPLAINS ONLY PART OF THE GAP. IS THE MODEL THEN "ALMOST FAIR"? WHAT SHOULD THE BANK DO?

## ✍️ Exercise — Remove the protected attribute: what happens to the gap?

1. Put the protected attribute in `features_to_drop`.
2. Re-run the two cells: we retrain, explain the same 300 applicants with SHAP (≈ 20 seconds) and compare the **decomposition of the gap**, before and after.

In [ ]:
# ==========================================
# 15. ✍️ EXERCISE: YOUR FIX
# ==========================================

features_to_drop = []        # 👉 e.g. ['feature_name']

kept = [f for f in FEATURES if f not in features_to_drop]
new_model = train_mortgage_model(X_train[kept], y_train)

def predict_new(data):
    return new_model.predict_proba(pd.DataFrame(data, columns=kept))[:, 1]

sv_new = shap.Explainer(predict_new, background[kept], algorithm="exact")(X_test[kept].iloc[:N_EXPLAIN])

In [ ]:
# ==========================================
# 16. THE GAP, BEFORE vs AFTER
# ==========================================

p_new = new_model.predict_proba(X_test[kept].iloc[:N_EXPLAIN])[:, 1]
new_gap = gap_by_feature(sv_new, kept).reindex(FEATURES).fillna(0)
table = pd.DataFrame({"Original model": gap, "New model": new_gap})
table.loc["TOTAL GAP"] = [pred_gap, p_new[is_afam].mean() - p_new[~is_afam].mean()]

ax = table.plot.barh(figsize=(8, 4.8), color=[GRAY, BLUE], width=0.75)
ax.invert_yaxis()
ax.axvline(0, color=INK, lw=0.8)
for bars in ax.containers:
    ax.bar_label(bars, fmt="%+.3f", padding=3, color=INK, fontsize=7)
ax.set_xlim(min(-0.015, table.values.min() - 0.01), table.values.max() * 1.15)
ax.set_xlabel("contribution to the gap in P(deny)")
ax.set_title("Gap between the groups, feature by feature", loc="left")
ax.legend(frameon=False, loc="upper right")
ax.xaxis.grid(True, color=GRID, lw=0.8); ax.set_axisbelow(True)
plt.show()
print(f"Dropped features: {features_to_drop}")

if not features_to_drop:
    print("ℹ️ features_to_drop is empty: the new model is identical to the original. Drop the protected attribute!")

<details>
<summary>✅ <b>Solution</b> (click to open)</summary>

```python
features_to_drop = ['afam']
```

- The contribution of `afam` disappears and the **total gap shrinks** by about the same amount.
- The contributions of the other features barely change: no single feature "takes the place" of race.
- The remaining gap comes from **real differences in the data** (loan-to-value, credit history, ...). Whether they are legitimate risk factors or the result of past discrimination is a question for economists, lawyers and regulators: SHAP tells us **where** to look.

</details>

# ⚠️ Part 4 — The limits of SHAP

- **Cost**: exact Shapley values need all $2^n$ teams: $2^9 = 512$ here, but $10^{15}$ with 50 features. In practice SHAP uses approximations or model-specific algorithms (e.g. `shap.TreeExplainer` for tree models).
- **Background**: SHAP explains *"why this prediction, compared with the background?"*. Change the background, and the explanation changes.
- **Correlated features**: filling the unknown features with values of other applicants can create unrealistic combinations.
- **Not causal**: SHAP explains the **model**, not the world. A large SHAP value means *"the model uses this feature"*, not *"this feature causes the outcome"*.

# 📌 Summary: LIME vs SHAP on the same mortgage model

| | LIME (Notebook 1) | SHAP (this notebook) |
| :--- | :--- | :--- |
| Idea | local linear approximation | fair split of the prediction (Shapley value) |
| Contributions add up to the prediction | ❌ | ✅ (efficiency) |
| Global view | ❌ one applicant at a time | ✅ aggregate the local values |
| Same result at every run | ❌ random sampling | ✅ (exact computation) |
| Reference point | the neighbourhood of $x$ | a background dataset |
| What it showed | race pushes applicant #112 toward denial | race adds a measurable share of the **gap** between groups |

### 🔑 The golden rule

> LIME and SHAP tell you **why** the model decides. Domain knowledge tells you whether that **why** is acceptable.

➡️ **Notebook 4** applies SHAP to **text**.

## 🎯 Final exercises

1. **Another applicant.** Draw the waterfall plot of an applicant who was **approved** and check the efficiency property by hand.
2. **Change the background.** Use 100 **approved** applicants as background (`X_train[y_train == 0].sample(100, random_state=42)`). How do the base value and the waterfall of applicant #112 change? Which question does SHAP answer now?
3. **A faster explainer.** Try `shap.TreeExplainer(mortgage_model)(X_test.iloc[:N_EXPLAIN])[:, :, 1]` (class 1 = denied), an algorithm specific to tree models. Compare its speed and its values with our exact explainer. Why are the values not identical? *(Hint: what is its reference point?)*
4. **Do LIME and SHAP agree?** For 20 applicants, compare the most important feature according to LIME and to SHAP. How often do they agree?

---

### References

- Shapley, L. S. (1953). *A Value for n-Person Games*. Contributions to the Theory of Games II, 307-317.
- Lundberg, S. M., Lee, S.-I. (2017). *A Unified Approach to Interpreting Model Predictions*. NeurIPS 2017.
- Lundberg, S. M. et al. (2020). *From local explanations to global understanding with explainable AI for trees*. Nature Machine Intelligence, 2, 56-67.
- Ribeiro, M. T., Singh, S., Guestrin, C. (2016). *"Why Should I Trust You?": Explaining the Predictions of Any Classifier*. KDD 2016.
- Molnar, C. *Interpretable Machine Learning* (free online book): <https://christophm.github.io/interpretable-ml-book/>
- Munnell, A. H., Tootell, G. M. B., Browne, L. E., McEneaney, J. (1996). *Mortgage Lending in Boston: Interpreting HMDA Data*. American Economic Review, 86(1), 25-53.
- Stock, J. H., Watson, M. W. *Introduction to Econometrics* (chapter on binary dependent variables, Boston HMDA data).
- SHAP library: <https://github.com/shap/shap>